## Features extraction from description 


In [87]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv(r"../Data/raw/airbnb_processed.csv", encoding="utf-8")
df.shape

(831, 27)

In [88]:
df[['id','description']].head()

,id,description
0,1292713234154945394,Enjoy your stay with Panoramic View of the Giz...
1,1508718511630646313,Welcome To Marhaba Pyramids View Hotel✨Wake up...
2,1297327219631789358,The place is spacious and can accommodate more...
3,1606001853199411128,A designer retreat where ancient soul meets mo...
4,1314833467096489875,Enjoy stay in Single RoomTHE ROOM FEATURES1 Qu...


In [89]:
df['description'] = df['description'].astype(str)

Calculate the description length

In [90]:
# Add a column with the description length
df['description_length'] = df['description'].apply(lambda x: len(str(x).split()))
df[['description', 'description_length']].head()

,description,description_length
0,Enjoy your stay with Panoramic View of the Giz...,328
1,Welcome To Marhaba Pyramids View Hotel✨Wake up...,80
2,The place is spacious and can accommodate more...,98
3,A designer retreat where ancient soul meets mo...,249
4,Enjoy stay in Single RoomTHE ROOM FEATURES1 Qu...,119


### Features Extraction
- `spacy` for extracting has_wifi, has_pyramids_view and has_pool boolean features


In [91]:
# spaCy/keyword-based extraction: WiFi and Pool features

def extract_bool_feature(text):
    if pd.isnull(text):
        return pd.Series({'has_wifi': None, 'has_pool': None})
    text_lower = str(text).lower()
    # WiFi detection
    wifi_keywords = ['wifi', 'wi-fi', 'wireless internet', 'internet access', 'free wifi', 'fast wifi', 'high-speed wifi']
    has_wifi = any(kw in text_lower for kw in wifi_keywords)
    # Pool detection
    pool_keywords = ['pool', 'swimming pool', 'private pool', 'outdoor pool', 'indoor pool', 'jacuzzi', 'hot tub']
    # Pyramid View
    pyramid_keywords = ['pyramid view', 'pyramids view', 'view of the pyramids']
    has_pool = any(kw in text_lower for kw in pool_keywords)
    has_pyramid_view = any(kw in text_lower for kw in pyramid_keywords)
    return pd.Series({'has_wifi': has_wifi, 'has_pool': has_pool, 'has_pyramid_view': has_pyramid_view})

# Apply to DataFrame
df[['has_wifi', 'has_pool', 'has_pyramid_view']] = df['description'].apply(extract_bool_feature)
df[['description', 'has_wifi', 'has_pool', 'has_pyramid_view']].head()

,description,has_wifi,has_pool,has_pyramid_view
0,Enjoy your stay with Panoramic View of the Giz...,True,True,False
1,Welcome To Marhaba Pyramids View Hotel✨Wake up...,False,True,True
2,The place is spacious and can accommodate more...,False,False,True
3,A designer retreat where ancient soul meets mo...,True,True,True
4,Enjoy stay in Single RoomTHE ROOM FEATURES1 Qu...,False,False,True


#### Extracting `location`

In [92]:
# # Extract location from description or title and classify into known areas
# import re

# def classify_location(text):
#     if pd.isnull(text):
#         return None
#     text = str(text).lower()
#     if re.search(r"giza|gîza", text):
#         return "Giza"
#     elif re.search(r"new cairo|newcairo|new-cairo", text):
#         return "New cairo"
#     elif re.search(r"maadi|ma'adi|el maadi", text):
#         return "Maadi"
#     elif re.search(r"zamalek", text):
#         return "Zamalek"
#     elif re.search(r"nazlet el semman|nazlet elsamman|nazlet al semman|nazlet al-samman", text):
#         return "Nazlet El Semman"
#     elif re.search(r"downtown cairo|downtown", text):
#         return "Downtown Cairo"
#     elif re.search(r"Garden City", text):
#         return "Garden city"
#     elif re.search(r"Cairo", text):
#         return "Cairo"
#     else:
#         return None

# # Apply to both description and title, prefer title if both present

# def extract_location(row):
#     loc = classify_location(row['title'])
#     if loc:
#         return loc
#     return classify_location(row['description'])

# df['classified_location'] = df.apply(extract_location, axis=1)
# df[['title', 'description', 'classified_location']]

In [94]:
# df['classified_location'].isnull().sum()


- Using `Llama 2` for extracting view features from the description

In [95]:
# import ollama
# import json
# import re
# import time

# # Define the extraction logic

# def extract_features(description, idx=None, total=None):
#     prompt = f"""
#     Extract ONLY the following from this rental description:
#     "{description}"

#     Return a JSON object with a key for the view.
#     The view is: The main view or location features
#     If a value is not mentioned, set it to null.
#     Output ONLY valid JSON with double quotes and no extra text.
#     """
#     start = time.time()
#     response = ollama.generate(
#         model='llama3.2:1b',
#         prompt=prompt,
#         format='json'
#     )
#     elapsed = time.time() - start
#     if idx is not None and total is not None:
#         print(f"[{idx+1}/{total}] Processed in {elapsed:.2f}s")
#     try:
#         match = re.search(r'\{.*\}', response['response'], re.DOTALL)
#         if match:
#             return json.loads(match.group(0))
#         else:
#             print("No JSON object found in response:", response['response'])
#             return {"view": None}
#     except Exception as e:
#         print(f"Error parsing JSON: {e}\nRaw response: {response['response']}")
#         return {"view": None}

# # Track progress and time for each row
# results = []
# total = len(df)
# start_time = time.time()
# for idx, desc in enumerate(df['description']):
#     features = extract_features(desc, idx, total)
#     results.append(features.get('view') if isinstance(features, dict) else None)
#     if (idx+1) % 10 == 0 or idx == total-1:
#         elapsed = time.time() - start_time
#         print(f"Progress: {idx+1}/{total} ({(idx+1)/total*100:.1f}%) - Elapsed: {elapsed/60:.2f} min")
# df['view'] = results
# df[['description', 'view']].head()


In [96]:
df.isnull().sum()

id                                   0
title                                0
url                                  0
thumbnail                            0
lat                                  0
lng                                  0
rating_overall                       0
reviews_count                        0
rating_accuracy                      0
rating_cleanliness                   0
rating_value                         0
rating_location                      0
description                          0
price_breakdown_baseprice_price      0
nightly_price                        0
price_price                        559
checkin_year                         0
checkin_month                        0
checkin_day                          0
checkin_weekday                      0
checkout_year                        0
checkout_month                       0
checkout_day                         0
checkout_weekday                     0
has_rating                           0
bedrooms                 

In [97]:
df.to_csv('../Data/raw/airbnb_processed.csv',index=False, encoding="utf-8-sig")

In [98]:
df.drop(columns=["description", "thumbnail","url", "id"], inplace=True)

In [99]:
df.to_csv('../Data/clean/airbnb_cleaned.csv',index=False, encoding="utf-8-sig")